<!--
File: technical_memory.ipynb
Description: Running technical decision log for The Village. Records architectural decisions, tradeoffs, and gotchas over the project's lifetime.
Author: Jose-Jorge HERNANDEZ
Company: Parlee Conseiller, Inc.
Date: 2026-09-07
Last edit date: 2026-10-15
Version: 4.0.0
-->

# Technical Memory — The Village

This notebook is a running log of *why* — design decisions, tradeoffs weighed and rejected, gotchas hit and how they were resolved. It is not a changelog of *what* changed (`CHANGELOG.md`) and not a bug log (`docs/BUGS.md`, once one exists) — see each project's own docs for those.

## 2026-09-07 — Documentation overhaul & project history recovery

### Context
The repo's docs (`README.md`, `TODO.md`, `ARCHITECTURE.md`, `CHANGELOG.md`, `GDD.md`) had drifted out of sync with both each other and the actual codebase, and with a much longer project history that lived entirely outside the git repo (in school coursework going back to Feb 2024).

### Decision 1 — Resolving the TODO.md vs CLAUDE.md phase-completion conflict
`CLAUDE.md` claimed "all 6 phases done" (movement/combat, rooms, items, main menu, pause menu, real sprites) while `TODO.md` still showed most of those as unchecked, using an older, differently-numbered phase scheme (0–8) that was never updated after those milestones actually shipped.

**Resolution:** verified directly against the file tree (`scenes/ui/MainMenu.tscn`, `GameOver.tscn`, `HealthUI.tscn`, `scenes/world/rooms/Room1_Tutorial.tscn` + `Room2_Challenge.tscn`, `scenes/items/HeartPickup.tscn`, wired sprites `tileset_forest.png`/`slime.png`/`sword.png`) — all exist. `CLAUDE.md` was correct; `TODO.md` was stale. Rewrote `TODO.md` onto `CLAUDE.md`'s phase numbering so the two documents can't drift apart the same way again.

### Decision 2 — Reconstructing an accurate CHANGELOG from git history instead of the vague `[Unreleased]` block
The changelog only had one dated entry (`[0.0.1]` — 2026-05-25) with everything else bucketed under `[Unreleased]`. Real commit history was available and more precise, so it was used to rebuild dated release entries.

### Decision 3 — Extended lore preserved as a separate `LORE.md`, not merged into `GDD.md`
School research (see below) surfaced a much richer original narrative (protagonist Joe, antagonist Caver, four dimensions) that the current `GDD.md` explicitly scopes *out* of the prototype. Rather than re-inflate the lean prototype GDD, the fuller narrative was captured in a new root-level `LORE.md`, marked as dormant/future-vision material — preserves it for later without disturbing the working, intentionally-small current scope.

### Decision 4 — Asset migration from the school folder, with a licensing caveat
Copied into `assets/branding/`, `assets/reference/joe/`, and `assets/audio/`:
- Branding art (`thevillage*.png/.ico`) and Joe's character reference art/pixel frames — both the student's own coursework, low risk.
- Audio clips (`bg_music`, `button_click`, `forest`, `person-running-loop-245173`, `retro-jump-3-236683`, `pianochocolate_forest`) made for the earlier Unreal Engine 5 prototype.

**Gotcha to resolve before shipping:** the audio files' actual licenses were not verified during this pass — `person-running-loop-245173` and `retro-jump-3-236683` look like they're from a stock SFX site (ID-suffixed filenames, e.g. Pixabay/Freesound convention) and `pianochocolate_forest.mp3` is named after a real artist ("Pianochocolate") and may not be freely redistributable. None of these should be included in a public/shipped build until their license terms are confirmed.

**Deliberately NOT copied:** `800609_warxwell_16x16-pixel-chars.png` (a third-party pixel-character reference sheet found alongside Joe's art) — this reads as someone else's published asset pack used only as visual reference, not the student's own work, so it was left in the school folder rather than copied into the repo.

In [ ]:
%%bash
# Command used to reconstruct the dated CHANGELOG entries from real commit history
# (run from the repo root)
git log --all --date=short --stat --pretty=format:"=== %ad | %s ===" --reverse

### Project history summary (for future sessions)

The Village did **not** start as this Godot repo. Full lineage, recovered from `H:\OneDrive\Office\School\UVM`:

| When | Course | What happened |
|---|---|---|
| 2024-02 | Estructuras Narrativas para Videojuegos | Concept originates — first GDD draft, gothic medieval village, isometric action-RPG |
| 2024-06 | Diseño de Personajes | Protagonist Joe fully designed — character sheet + 8-frame pixel sprite |
| 2024-08 | Diseño de Escena y Componentes | Worldbuilding expands — level map, moodboards, HUD/item art |
| 2024-11 to 2024-12 | Motor de Videojuegos | Built as a working Unreal Engine 5 prototype across 9 graded assignments (movement, collision, rigging, level blockout, menu, sound, AI, save system) |
| *(~17-month gap, no documented activity)* | | |
| 2026-05-25 → present | — | Rebuilt from scratch in Godot 4.6, deliberately scoped down to a lean top-down prototype (no story, single enemy type, no saves) — this repo |

The current repo is a clean-slate reboot in a new engine, not a continuation of the UE5 codebase — no code or assets carried over directly (the UE5 project files were never located in the school folder, only its documentation, videos, and a handful of standalone art/audio assets).

## 2026-10-15 — Rebuild on the Chapel of Dreams architecture (Phases 0 and 1)

### Context
The Godot prototype (two rooms, slimes, placeholder Joe) had been dormant since September. Meanwhile game 2 of the trilogy, *Chapel of Dreams*, was built to beta with a much stronger method: generated art from one palette, manifests read at runtime, `PlayerTuning`-style constant files, and headless scripted test suites. The owner asked to build this game for real, "same mechanics, visuals" as *A Link to the Past*.

### Decisions (owner-confirmed where noted)
1. **Rebuild in place, not extend** (owner). The prototype's autoload web (EventBus/GameManager/SceneManager/InputHandler) did not fit Chapel's patterns, and its art was placeholder. Rebuilding on Chapel's base reuses ~1,000 lines of proven code (HitBox/HurtBox/Health/StateMachine/SpriteStrips/HitFlash) and its whole tooling. The prototype stays in git history (`cec47ca`).
2. **384×216, not ALttP's 256×224** (owner). Consistency across the trilogy won over authenticity. Consequence: a screen is 24×13.5 tiles, so Phase 3's screen-scrolling areas will be 24×14 tiles with half a row hidden behind the HUD band, or the HUD overlays the field — to decide in Phase 3.
3. **Light/Dark mirror worlds** (owner). The four lore dimensions become Dark World regions; `palette.py` already carries a `DW_*` set so the Dark World can be the same generators with a palette swap.
4. **Feet-origin actors with a feet-only collision box** (10×6). This is how ALttP lets Link's head overlap the tile above a wall, and it makes y-sorting trivial (sort by origin).
5. **48×48 frames for Joe.** The sword sweeps around him; drawing it into the frame (instead of a separate rotated sprite) keeps the art exact and lets the manifest carry the hit window.
6. **Runtime TileSet from a JSON manifest** instead of a hand-maintained `.tres`: the generator is the single source of truth for which tiles are solid.
7. **Corner nudge via `test_move`**: when a single-axis step is fully blocked, probe 1..6 px sideways; the first clear offset slides Joe that way at walk speed. Diagonal input never nudges (the slide already handles it).

### Gotchas met
- Case-insensitive filesystem: `Tools/` vs `tools/` produced a *file* named `tools` (see `docs/BUGS.md`).
- `Input.action_press` is seen on the next physics frame: tests wait 2 frames.
- Naming a Node method `_set` would override `Object._set`.


In [ ]:
# The corner-nudge core (src/player/Player.gd, v1.0.0), kept here because its tuning is a feel decision.
func _try_corner_nudge(step: Vector2, delta: float) -> void:
	var perp := Vector2(step.y, step.x).abs()
	for k in range(1, PlayerTuning.CORNER_NUDGE_PX + 1):
		for side in [-1.0, 1.0]:
			var off: Vector2 = perp * side * float(k)
			if not test_move(global_transform.translated(off), step):
				move_and_collide(perp * side * PlayerTuning.NUDGE_SPEED * delta)
				return


## 2026-10-15 — Phase 2: combat core

### Decisions
1. **Shield as a passive cone, not a button.** ALttP's shield blocks automatically while Link faces the projectile. Joe's `_on_hurt` asks `HitBox.blockable` and `blocks_from(source)` (dot product with the facing >= 0.6, i.e. a ~106 degree cone) and only in `idle`/`walk`/`charge`. Swinging, spinning, lifting or carrying leaves him open — the same trade-off ALttP makes.
2. **Projectiles tell the target, the target tells the projectile.** `HurtBox` emits `hurt` *before* `HitBox.landed`, so Joe can call `deflect()` on the rock first and the rock's own `landed` handler then sees `deflected` and stays alive to bounce away. No extra signal needed.
3. **Hit-stop is per actor (`freeze`), not `Engine.time_scale`.** Kept from Chapel: it stays deterministic in tests and does not stall the camera shake, HUD or other actors. Joe pauses his `StateMachine` node's physics process during a freeze.
4. **Thrown objects move on the ground, only the sprite flies.** The body and HitBox travel at ground level (walls and enemies are hit where they stand) while the sprite follows a falling parabola from carry height — ALttP's model, and it keeps collision honest.
5. **Spin as one shared strip with per-facing start frames.** Frame *i* holds the sword at i×45°; the manifest maps each facing to its first frame. One 8-frame strip instead of four, and `flip_h` is turned off for the spin because the left-facing frames are already drawn mirrored.
6. **Enemies overlap Joe and each other.** Like ALttP, enemy bodies collide only with the world and props; damage is by touch. It avoids enemies body-blocking Joe in corridors.
7. **A minimal HUD now (hearts + gems) instead of in Phase 4.** Without it, damage and drops were invisible to a playtester; the full HUD remains Phase 4.

### Gotchas
- In `-s` test scripts the autoloads are reached with `root.get_node(...)`; a static reference compiles before they exist (Chapel lesson, applied up front, no failure hit).
- A windowed capture with live guards produced shots of Joe blinking under a guard: captures now switch his hurt box off.


In [ ]:
# Shield check (src/player/Player.gd v2.0.0)
func blocks_from(source_position: Vector2) -> bool:
	if not fsm.current_state in PlayerTuning.SHIELD_STATES:
		return false
	var to_src := source_position - (global_position + PlayerTuning.BODY_CENTER)
	if to_src.length() < 0.01:
		return false
	return to_src.normalized().dot(Vector2(facing)) >= PlayerTuning.SHIELD_DOT


## 2026-10-15 — Phase 3: overworld engine

### Decisions
1. **One global coordinate system for the overworld.** Each area JSON carries `origin_tiles`; neighbours share edges. Crossing an edge needs no coordinate translation: Joe keeps walking, the World notices his feet left the current rect and asks `overworld_at(feet)` which area owns that point. This also made the lint simple (a global tile -> area map).
2. **Only one area alive at a time** (two during the 0.6 s slide). Cheap, and it gives ALttP's behaviour for free: enemies and bushes reset when you come back.
3. **The camera belongs to the World, not to Joe.** It is clamped per area, centred in rooms smaller than the view, and tweened during scrolls. Test arenas keep the simple child camera.
4. **Ledges are solid tiles plus a flag.** Rather than one-way collision (which Godot defines along a body's local up, awkward top-down), a ledge cell is solid and carries a `ledge` custom-data flag; walking south into it switches Joe to a `hop` state that flies him over the ledge cells without collision. The lint's reachability search understands the one-way edge.
5. **Doors are walkable cells in the house's footprint**, triggered by Joe's feet entering the cell while facing up; exit mats trigger when facing down. No Area2D triggers to keep in sync with the tiles.
6. **Generator owns pairing.** `gen_areas.py` links each house door with its interior's mat in both directions, so a door can never lead to a room without a way back (and the lint re-checks it).
7. **Screen = 24x14 tiles.** 216 px is 13.5 tiles; a 14-tile screen lets the camera scroll 8 px inside one screen rather than leaving half-tiles at area edges.

### Gotchas
- A new `class_name` made the headless suite hang (not fail) until `--import` ran; all Godot runs are now wrapped in `timeout`.
- Nine-slice autotiling must treat off-area cells as the same terrain, or every road shows a grass lip at area seams.
- My first capture script released the stick one frame before Joe reached the edge - a reminder that a windowed shot that "shows nothing happened" needs the timing checked before the code is blamed.


In [ ]:
# How the World finds where Joe went when he leaves an area (src/world/World.gd v1.0.0)
func overworld_at(p: Vector2) -> String:
	for aid in areas:
		if areas[aid]["kind"] == "overworld" and rect_of(aid).has_point(p):
			return aid
	return ""
